# PII-Safe Customer Support Assistant

Draft replies to Indian customer-support tickets with `sarvam-105b` **without sending the customer's
personal data to the model**. Aadhaar, PAN, UPI IDs, phone numbers, names and emails are swapped for
typed placeholders such as `<AADHAAR_1>` before the request leaves your machine, and the real values are
put back into the model's reply afterwards.

Pipeline overview:
1. Mask the ticket locally with [MaskFlow](https://github.com/maskflow/maskflow) (open source, MIT)
2. Send only the masked text to Sarvam's Chat Completions API (`sarvam-105b`)
3. Restore the original values in the reply, locally
4. Keep placeholders stable across a multi-turn conversation
5. Save the masked transcript (what the model actually saw) to `outputs/`

In [ ]:
%pip install -r requirements.txt

## Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import maskflow
from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Why mask before the API call?

Support tickets in India routinely contain Aadhaar numbers, PANs, UPI IDs and phone numbers. Under the
Digital Personal Data Protection (DPDP) Act, sending them to any third-party API is a data-sharing
decision you have to be able to justify. Masking locally means the model still gets everything it needs
to write a good reply (it knows *that* there is an Aadhaar number, not *what* it is), and the raw values
never leave your process.

MaskFlow validates checksums where a public one exists (Verhoeff for Aadhaar, mod-36 for GSTIN), so a
random 12-digit order number is not mistaken for an Aadhaar number.

All personal data below is **synthetic**. The Aadhaar number is generated to pass the Verhoeff checksum
and does not belong to anyone.

In [ ]:
# Synthetic ticket. Every identifier is made up; the Aadhaar number is a
# generated, checksum-valid value that belongs to no one.
ticket = (
    "Namaste, main Priya Sharma hoon. Mera Aadhaar 2346 8907 6549 aur PAN ABCPS1234F hai. "
    "UPI priya.sharma@okhdfcbank se Rs 4,500 kat gaye par refund nahi aaya. "
    "Mujhe +91-9876543210 par call karein ya priya.s@example.in par email karein."
)
print(ticket)

## Step 1: Mask the ticket locally

A MaskFlow **session** keeps the mapping from placeholders to real values in memory, on your machine.
`patterns_only=True` uses MaskFlow's pattern, checksum and name-list detectors and skips the spaCy NER
model, so there is nothing extra to download.

In [ ]:
session = maskflow.session(patterns_only=True)

masked_ticket = session.mask(ticket)
print(masked_ticket)

# Nothing sensitive is left in what we are about to send.
for raw in ["2346 8907 6549", "ABCPS1234F", "priya.sharma@okhdfcbank", "9876543210"]:
    assert raw not in masked_ticket

## Step 2: Ask `sarvam-105b` for a reply, using only the masked text

The system prompt asks the model to copy placeholders exactly as written, so they can be swapped back
afterwards.

In [ ]:
SYSTEM_PROMPT = (
    "You are a polite customer-support agent for an Indian payments app. "
    "Reply in the same language and style the customer used. "
    "The ticket contains placeholders such as <PERSON_NAME_1> or <UPI_VPA_1> that stand in for "
    "the customer's personal details. Copy any placeholder you need exactly as written, "
    "including the angle brackets, and never invent new ones."
)

history: list[dict[str, str]] = [{"role": "system", "content": SYSTEM_PROMPT}]


def ask_sarvam(masked_message: str) -> str:
    """Send one masked user turn to sarvam-105b and return its masked reply."""
    history.append({"role": "user", "content": masked_message})
    response = client.chat.completions(model="sarvam-105b", messages=history)
    reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply


masked_reply = ask_sarvam(masked_ticket)
print("What sarvam-105b returned:\n")
print(masked_reply)

## Step 3: Restore the real values, locally

In [ ]:
reply = session.unmask(masked_reply)
print(reply)

## Step 4: Multi-turn conversations keep the same placeholders

Within a session, the same value always gets the same placeholder. When the customer repeats their
Aadhaar number in a follow-up, the model sees `<AADHAAR_1>` again and can tell it is the same person.

In [ ]:
follow_up = (
    "Maine apna Aadhaar 2346 8907 6549 aapke app par verify bhi kar diya tha. "
    "Refund kab tak aayega?"
)

masked_follow_up = session.mask(follow_up)
print(masked_follow_up)
assert "<AADHAAR_1>" in masked_follow_up

print()
print(session.unmask(ask_sarvam(masked_follow_up)))

## Step 5: Save what the model actually saw

The transcript written to `outputs/` contains only masked text, which makes it safe to keep as a record
of exactly what was shared with the API. Closing the session discards the placeholder mapping.

In [ ]:
transcript_path = OUTPUT_DIR / "masked_transcript.json"
transcript_path.write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {len(history)} masked messages to {transcript_path}")

session.close()

## Limits and next steps

- **Names in Devanagari script are not detected yet.** Identifiers (Aadhaar, PAN, UPI, phone, email) are
  detected whatever script the surrounding text is in, but a name written as प्रिया शर्मा passes through
  unmasked. Names written in Latin script, as in Hinglish, are detected.
- `patterns_only=True` trades some name and address recall for speed and zero model downloads. Drop it
  (and run `python -m spacy download en_core_web_sm`) to add spaCy's NER pass.
- Per-entity accuracy, including the cases MaskFlow misses, is measured and published in the
  [MaskFlow benchmark](https://github.com/maskflow/maskflow#benchmark).
- The same masking is available as a LiteLLM guardrail, a LangChain/LlamaIndex component, an MCP proxy,
  and a drop-in OpenAI-compatible gateway: see the [MaskFlow repository](https://github.com/maskflow/maskflow).
- Sarvam docs: [Chat Completions API](https://docs.sarvam.ai/api-reference-docs/chat/completions).